In [272]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_diabetes, load_breast_cancer, load_iris, load_digits
from sklearn.model_selection import train_test_split

Step 1: Datasets loading and splitting

In [273]:
#Diabetes dataset
diabetes=load_diabetes(scaled=False) #this returns an object called Bunch,which is similar to a dictionary.
x_diabetes=diabetes.data #features
y_diabetes=diabetes.target #target

x_diabetes_train, x_diabetes_test, y_diabetes_train, y_diabetes_test = train_test_split(x_diabetes, y_diabetes, test_size=0.2, random_state=42)
x_diabetes_train, x_diabetes_validation, y_diabetes_train, y_diabetes_validation = train_test_split(x_diabetes_train, y_diabetes_train, test_size=0.25, random_state=42)

#Diabetes is a regression dataset, so we will not stratify the target variable when splitting the data.
#Stratifcation means that target is distributed evenly between the train, validation and test sets.

In [274]:
#Breast cancer dataset
cancer=load_breast_cancer() #this returns an object called Bunch,which is similar to a dictionary.
x_cancer=cancer.data #features
y_cancer=1-cancer.target #target

x_cancer_train, x_cancer_test, y_cancer_train, y_cancer_test = train_test_split(x_cancer, y_cancer, test_size=0.2, random_state=42, stratify=y_cancer)
x_cancer_train, x_cancer_validation, y_cancer_train, y_cancer_validation = train_test_split(x_cancer_train, y_cancer_train, test_size=0.25, random_state=42,stratify=y_cancer_train)


In [275]:
#Iris dataset
iris=load_iris() #this returns an object called Bunch,which is similar to a dictionary.
x_iris=iris.data #features      
y_iris=iris.target #target

x_iris_train, x_iris_test, y_iris_train, y_iris_test = train_test_split(x_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris)
x_iris_train, x_iris_validation, y_iris_train, y_iris_validation = train_test_split(x_iris_train, y_iris_train, test_size=0.25, random_state=42,stratify=y_iris_train)

In [276]:
#Digits dataset
digits=load_digits() #this returns an object called Bunch,which is similar to a dictionary.
x_digits=digits.data//16 #features
y_digits=digits.target #target

x_digits_train, x_digits_test, y_digits_train, y_digits_test = train_test_split(x_digits, y_digits, test_size=0.2, random_state=42, stratify=y_digits)
x_digits_train, x_digits_validation, y_digits_train, y_digits_validation = train_test_split(x_digits_train, y_digits_train, test_size=0.25, random_state=42,stratify=y_digits_train)

In [277]:
data= {
    "Train":[x_diabetes_train.shape[0], x_cancer_train.shape[0], x_digits_train.shape[0], x_iris_train.shape[0]],
    "Validation":[x_diabetes_validation.shape[0], x_cancer_validation.shape[0], x_digits_validation.shape[0], x_iris_validation.shape[0]],
    "Test":[x_diabetes_test.shape[0], x_cancer_test.shape[0], x_digits_test.shape[0], x_iris_test.shape[0]],
    "Total":[x_diabetes.shape[0], x_cancer.shape[0], x_digits.shape[0], x_iris.shape[0]]
}   #shape[0] gives the number of rows in the dataset, which is the number of samples.

table=pd.DataFrame(data ,index=["Diabetes", "Breast Cancer", "Digits", "Iris"])
print(table)

               Train  Validation  Test  Total
Diabetes         264          89    89    442
Breast Cancer    341         114   114    569
Digits          1077         360   360   1797
Iris              90          30    30    150


This is the number of samples in the training, validation and test sets for each dataset, proving that they match the table provided in the pdf.

Diabetes: B. Efron, T. Hastie, I. Johnstone, R. Tibshirani, “Least Angle Regression,” 
Annals of Statistics 32(2), 2004. Breast cancer: W. N. Street, W. H. Wolberg, O. L. Mangasarian, “Nuclear feature 
extraction for breast tumor diagnosis,” 1993, UCI Machine Learning Repository (archive.ics.uci.edu/dataset/17). 
Digits: E. Alpaydin, C. Kaynak, “Optical Recognition of Handwritten Digits,” UCI Machine Learning Repository 
(archive.ics.uci.edu/dataset/80). Iris: R. A. Fisher, “The use of multiple measurements in taxonomic problems,” 
Annals of Eugenics 7(2), 1936 (archive.ics.uci.edu/dataset/53). 

Step2: Standardisation and adding bias

In [278]:
def Standardizer(x,x_training):
    #Get the mean and standard deviation from training dataset only
    #Use it to standardize the training, validation and test datasets.
    x_standardized=np.zeros(x.shape)
    x_standardized_training=np.zeros(x_training.shape)
    number_of_features=x_training.shape[1]

    for i in range(number_of_features):
        mean=x_training[:,i].mean()
        std=x_training[:,i].std()
        if std==0:
            std=1
        x_standardized_training[:,i]=(x_training[:,i]-mean)/std
        x_standardized[:,i]=(x[:,i]-mean)/std
        print(f"Feature {i+1} standardized: mean={x_standardized_training[:,i].mean()}, std={x_standardized_training[:,i].std()}")
    
    return x_standardized


Mean and standard deviation are calculated from the training dataset only because otherwise the model would learn from the validation and test datasets which is invalid.

In [279]:
def add_bias(x):
    #Add a column of ones to the dataset to be multiplied by bias term in linear regression.
    ones=np.ones(x.shape[0],dtype=x.dtype)
    x_with_bias=np.column_stack((ones,x)) #pad ones to the left side.
    return x_with_bias

Step3: Error Calculations

Regression Metrics:

In [280]:
def mse(y_true, y_pred):
    #Mean Squared Error
    return np.mean((y_true-y_pred)**2)

In [281]:
def rmse(y_true, y_predicted):
    #Root Mean Squared Error
    return np.sqrt(mse(y_true, y_predicted))

In [282]:
def r_squared(y_true, y_predicted):
    #coefficient of determination
    ss_res=np.sum((y_true-y_predicted)**2)
    ss_tot=np.sum((y_true-np.mean(y_true))**2)
    return 1-(ss_res/ss_tot)

Classification Metrics:

In [283]:
def accuracy(y_true, y_predicted):
    return np.mean(y_true==y_predicted)

In [284]:
def confusion_matrix(y_true,y_predicted,k):
    confusion=np.zeros((k,k))
    true_positive, true_negative, false_positive, false_negative=0,0,0,0
    for i in range(y_true.size):
        confusion[int(y_true[i]),int(y_predicted[i])]+=1
    return confusion

In [285]:
def my_precision(y_true,y_predicted,class_label):
    
    confusion=confusion_matrix(y_true,y_predicted,int(np.max(y_true))+1)
    true_positive=confusion[class_label,class_label]
    false_positive=np.sum(confusion[:,class_label])-true_positive

    #This is for the class that is never predicted:
    if true_positive+false_positive==0: 
        return 0 
    
    return true_positive/(true_positive+false_positive)

In [286]:
def my_recall(y_true,y_predicted,class_label):
    
    confusion=confusion_matrix(y_true,y_predicted,int(np.max(y_true))+1)
    true_positive=confusion[class_label,class_label]
    false_negative=np.sum(confusion[class_label,:])-true_positive
    
    return true_positive/(true_positive+false_negative)

In [287]:
def my_f1_score(y_true,y_predicted,class_label):
    p=my_precision(y_true,y_predicted,class_label)
    r=my_recall(y_true,y_predicted,class_label)
    f1=2*(p*r)/(p+r)
    return f1

In [288]:
def macro_precision(y_true,y_predicted):
    k=int(np.max(y_true))+1
    precision_sum=0
    for i in range(k):
        precision_sum+=my_precision(y_true,y_predicted,i)
    return precision_sum/k

In [289]:
def macro_recall(y_true,y_predicted):
    k=int(np.max(y_true))+1
    recall_sum=0
    for i in range(k):
        recall_sum+=my_recall(y_true,y_predicted,i)
    return recall_sum/k

In [290]:
def macro_f1_score(y_true,y_predicted):
    k=int(np.max(y_true))+1
    f1_score_sum=0
    for i in range(k):
        f1_score_sum+=my_f1_score(y_true,y_predicted,i)
    return f1_score_sum/k

Testing the metric functions:

In [291]:
from sklearn.metrics import precision_score, recall_score, f1_score

actual_array=np.zeros(500)
predicted_array=np.zeros(500)
for i in range(500):
    actual_array[i]=np.random.randint(0,4)
    predicted_array[i]=np.random.randint(0,4)
print("My Functions: \n")
print("Macro Precision: ", macro_precision(actual_array, predicted_array))
print("Macro Recall: ", macro_recall(actual_array, predicted_array))
print("Macro F1 Score: ", macro_f1_score(actual_array, predicted_array))

print("\nSklearn Functions: \n")
print("Macro Precision: ", precision_score(actual_array, predicted_array, average='macro'))
print("Macro Recall: ", recall_score(actual_array, predicted_array, average='macro'))
print("Macro F1 Score: ", f1_score(actual_array, predicted_array, average='macro'))



My Functions: 

Macro Precision:  0.23806628379328554
Macro Recall:  0.23877362930425916
Macro F1 Score:  0.23826367437403947

Sklearn Functions: 

Macro Precision:  0.23806628379328554
Macro Recall:  0.23877362930425916
Macro F1 Score:  0.23826367437403947


In [ ]:
def grad_check(loss,gradient,weights):
    analytical_gradient=gradient(weights)
    if loss==hinge_loss:
        epsilon=1e-6
    else:
        epsilon=1e-5
    numerical_gradient=np.zeros(weights.shape)
    for index in np.ndindex(weights.shape):
        weights_plus=weights.copy()
        weights_plus[index]+=epsilon
        weights_minus=weights.copy()
        weights_minus[index]-=epsilon
        numerical_gradient[index]=(loss((weights_plus))-loss((weights_minus)))/(2*epsilon)
    error=np.linalg.norm(analytical_gradient-numerical_gradient)/max(np.linalg.norm(analytical_gradient),np.linalg.norm(numerical_gradient),1e-12)
    return error
        
    

In [ ]:
def hinge_loss(y_true, y_pred):
    #Hinge loss for binary classification
    return np.mean(np.maximum(0, 1 - y_true * y_pred))